<a id="intro"></a>
# ЛР № 5. Расчёт метрик кинофильмов (Movie Industry, 1980–2019)

Первая часть общего аналитического ноутбука ЛР № 5 и № 7. Здесь читаем готовую витрину `movies.parquet` из слоя `mart`, рассчитываем метрики основного вопроса проекта, формируем три результата в памяти DuckDB и проверяем согласованность чисел. В ЛР № 7 ниже будут добавлены графики и визуальный отчёт; промежуточные файлы данных на S3 или диске не создаются.

- Описание метрик, окна, подготовленных таблиц, проверок и выводы: [docs/metrics.md](../docs/metrics.md).
- Модель витрины ЛР № 4: [docs/mart.md](../docs/mart.md).
- Постановка задачи: [docs/project_brief.md](../docs/project_brief.md).

## 1. Вход и параметры

Используем витрину `s3://mart/movie_industry/source_ingested_on=2026-10-24/movies.parquet`, сохранённую в ЛР № 4 за целевой период четырёх полных десятилетий `1980–2019` гг.

In [1]:
# M01. Параметры
DATASET_SLUG = "movie_industry"
SOURCE_INGESTED_ON = "2026-10-24"
SNAPSHOT_PREFIX = f"{DATASET_SLUG}/source_ingested_on={SOURCE_INGESTED_ON}"
MART_PATH = f"s3://mart/{SNAPSHOT_PREFIX}/movies.parquet"
STUDY_YEAR_MIN = 1980
STUDY_YEAR_MAX = 2019
print(f"Вход: {MART_PATH}")
print(f"Период по release_year: [{STUDY_YEAR_MIN}; {STUDY_YEAR_MAX}]")

Вход: s3://mart/movie_industry/source_ingested_on=2026-10-24/movies.parquet
Период по release_year: [1980; 2019]


## 2. Подключение

Переиспользуем настройку S3 из предыдущих лабораторных работ. Вспомогательная функция `select_table` только выполняет переданный SQL-запрос и отображает HTML-таблицу с числом строк; вся логика расчётов записана в явных SQL-запросах.

In [2]:
# M02. Подключение
import os
import sys
from html import escape
from pathlib import Path

import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
    and (parent / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=False)
required_env = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing_env = [name for name in required_env if not os.getenv(name)]
if missing_env:
    msg = "Не заданы переменные окружения S3: " + ", ".join(missing_env)
    raise RuntimeError(msg)

src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
from analysis.read_raw import configure_s3  # noqa: E402

connection = duckdb.connect()
configure_s3(connection)


def literal(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


def select_table(sql: str) -> list[tuple[object, ...]]:
    cursor = connection.execute(sql)
    description = cursor.description
    if description is None:
        raise ValueError("Ожидается таблица результата")
    rows = cursor.fetchall()
    header = "".join(f"<th>{escape(str(col[0]))}</th>" for col in description)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead>"
                 f"<tbody>{body}</tbody></table>"))
    print(f"Строк результата: {len(rows)}")
    return rows

## 3. Чтение витрины

Одна строка таблицы `movies` представляет один полнометражный кинофильм за `1980–2019` гг. с составным ключом `(lower(movie_title), release_year)`. Таблица `analysis_base` один раз материализует данные витрины из S3 в оперативной памяти DuckDB с сохранением всех `7 640` фильмов, включая записи с предупреждающими флагами ЛР № 3.

In [3]:
# M03. Вход
connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW movies_mart AS
    SELECT * FROM read_parquet({literal(MART_PATH)}, hive_partitioning = false)
""")
required_fields = {
    "movie_title",
    "release_year",
    "release_decade",
    "genre",
    "rating",
    "imdb_score",
    "imdb_votes",
    "budget_usd",
    "gross_usd",
    "gross_to_budget_ratio",
    "is_gross_ge_budget",
    "incomplete_financials_flag",
    "nonstandard_rating_flag",
}
schema = {row[0]: row[1] for row in connection.execute("DESCRIBE movies_mart").fetchall()}
assert not (required_fields - schema.keys()), "В витрине отсутствуют нужные поля"

connection.execute(f"""
    CREATE OR REPLACE TEMP TABLE analysis_base AS
    SELECT *
    FROM movies_mart
    WHERE release_year BETWEEN {STUDY_YEAR_MIN} AND {STUDY_YEAR_MAX}
""")
input_rows = select_table("""
    SELECT
        count(*) AS films_count,
        count(*) FILTER (WHERE incomplete_financials_flag) AS incomplete_fin_count,
        count(*) FILTER (WHERE NOT incomplete_financials_flag) AS complete_fin_count,
        count(*) FILTER (WHERE is_gross_ge_budget) AS recouped_count,
        count(*) FILTER (WHERE nonstandard_rating_flag) AS nonstandard_rating_count,
        min(release_year) AS min_year,
        max(release_year) AS max_year
    FROM analysis_base
""")
print(f"Физических полей витрины: {len(schema)}")

films_count,incomplete_fin_count,complete_fin_count,recouped_count,nonstandard_rating_count,min_year,max_year
7640,2213,5427,3677,419,1980,2019


Строк результата: 1
Физических полей витрины: 13


<a id="definitions"></a>
## 4. Определения рассчитываемых метрик

По основному вопросу `Project Brief` рассчитываем четыре метрики:
- `films_count` — общее число выпущенных фильмов (`count(*)`, все `7 640` записей);
- `median_return_ratio` — медианный коэффициент кассовой отдачи `median(gross_to_budget_ratio)` среди фильмов с полной положительной финансовой парой (`NOT incomplete_financials_flag`, `5 427` записей);
- `recoup_rate_pct` — доля кассово окупившихся релизов (`gross_usd >= budget_usd`) в `%` от числа фильмов с полной финансовой парой (`complete_fin_count`);
- `median_imdb_score` — медианная оценка зрителей IMDb `median(imdb_score)` по всем фильмам группы.

Подробные правила участия строк и описание оконной функции приведены в [docs/metrics.md](../docs/metrics.md).

## 5. Общий итог

Отбор полных финансовых пар выполняется через `FILTER (WHERE ...)` и учёт `NULL` в производных полях, чтобы общий `WHERE` не исключал `2 213` фильмов без бюджета или сборов из подсчёта `films_count` и `median_imdb_score`.

In [4]:
# M04. Общие метрики
connection.execute("""
    CREATE OR REPLACE TEMP TABLE overall_metrics AS
    SELECT
        count(*) AS films_count,
        count(*) FILTER (
            WHERE NOT incomplete_financials_flag
        ) AS complete_fin_count,
        count(*) FILTER (WHERE is_gross_ge_budget) AS recouped_count,
        median(gross_to_budget_ratio) AS median_return_ratio,
        100.0 * count(*) FILTER (WHERE is_gross_ge_budget)
            / nullif(
                count(*) FILTER (WHERE NOT incomplete_financials_flag), 0
            ) AS recoup_rate_pct,
        median(imdb_score) AS median_imdb_score
    FROM analysis_base
""")
overall_rows = select_table("""
    SELECT
        films_count,
        complete_fin_count,
        recouped_count,
        round(median_return_ratio, 2) AS median_return_ratio,
        round(recoup_rate_pct, 2) AS recoup_rate_pct,
        round(median_imdb_score, 2) AS median_imdb_score
    FROM overall_metrics
""")
overall_raw = connection.execute("SELECT * FROM overall_metrics").fetchone()
assert overall_raw is not None

films_count,complete_fin_count,recouped_count,median_return_ratio,recoup_rate_pct,median_imdb_score
7640,5427,3677,1.83,67.75,6.5


Строк результата: 1


## 6. Основная группировка по жанрам и оконная функция

Одна строка таблицы `genre_metrics` представляет один основной жанр (`genre`) за весь период `1980–2019` гг. (`19` жанров). Оконная функция добавляет `percent_of_films` — долю каждого жанра в общем числе фильмов (`100.0 * films_count / nullif(sum(films_count) OVER (), 0)`).

In [5]:
# M05. Основная группировка по жанрам и окно
connection.execute("""
    CREATE OR REPLACE TEMP TABLE genre_metrics AS
    WITH grouped AS (
        SELECT
            genre,
            count(*) AS films_count,
            count(*) FILTER (
                WHERE NOT incomplete_financials_flag
            ) AS complete_fin_count,
            count(*) FILTER (WHERE is_gross_ge_budget) AS recouped_count,
            median(gross_to_budget_ratio) AS median_return_ratio,
            100.0 * count(*) FILTER (WHERE is_gross_ge_budget)
                / nullif(
                    count(*) FILTER (WHERE NOT incomplete_financials_flag), 0
                ) AS recoup_rate_pct,
            median(imdb_score) AS median_imdb_score
        FROM analysis_base
        GROUP BY genre
    )
    SELECT
        *,
        100.0 * films_count
            / nullif(sum(films_count) OVER (), 0) AS percent_of_films
    FROM grouped
""")
genre_rows = select_table("""
    SELECT
        genre,
        films_count,
        complete_fin_count,
        recouped_count,
        round(median_return_ratio, 2) AS median_return_ratio,
        round(recoup_rate_pct, 2) AS recoup_rate_pct,
        round(median_imdb_score, 2) AS median_imdb_score,
        round(percent_of_films, 3) AS percent_of_films
    FROM genre_metrics
    ORDER BY films_count DESC, genre
""")
genre_raw = connection.execute("""
    SELECT *
    FROM genre_metrics
    ORDER BY films_count DESC, genre
""").fetchall()

genre,films_count,complete_fin_count,recouped_count,median_return_ratio,recoup_rate_pct,median_imdb_score,percent_of_films
Comedy,2242,1496,1001,1.79,66.91,6.3,29.346
Action,1697,1411,1025,1.87,72.64,6.2,22.212
Drama,1510,868,522,1.52,60.14,6.7,19.764
Crime,549,400,224,1.31,56.0,6.7,7.186
Biography,443,312,201,1.48,64.42,7.1,5.798
Adventure,425,325,214,1.9,65.85,6.4,5.563
Animation,336,278,235,2.93,84.53,6.8,4.398
Horror,320,254,201,3.05,79.13,5.8,4.188
Fantasy,44,42,30,2.14,71.43,6.05,0.576
Mystery,20,17,11,1.98,64.71,6.65,0.262


Строк результата: 19


## 7. Уточняющая группировка по десятилетиям и жанрам и отдельные значения

Уточняем жанровую картину по четырём полным десятилетиям выпуска (`release_decade`, `genre`) в таблице `segment_metrics` (`62` фактически представленных сочетания; отсутствующие пары не заполняются искусственными нулями). Дополнительно выводим сводку по четырём десятилетиям и сохраняем в `distribution_values` (`7 640` строк) индивидуальные числовые значения `imdb_score`, `gross_to_budget_ratio`, `budget_usd` и `gross_usd` для построения распределений и анализа связи в ЛР № 7.

In [6]:
# M06. Уточняющая группировка и отдельные значения для продолжения анализа
connection.execute("""
    CREATE OR REPLACE TEMP TABLE segment_metrics AS
    SELECT
        release_decade,
        genre,
        count(*) AS films_count,
        count(*) FILTER (WHERE NOT incomplete_financials_flag) AS complete_fin_count,
        count(*) FILTER (WHERE is_gross_ge_budget) AS recouped_count,
        median(gross_to_budget_ratio) AS median_return_ratio,
        100.0 * count(*) FILTER (WHERE is_gross_ge_budget)
            / nullif(
                count(*) FILTER (WHERE NOT incomplete_financials_flag), 0
            ) AS recoup_rate_pct,
        median(imdb_score) AS median_imdb_score
    FROM analysis_base
    GROUP BY release_decade, genre
""")
print("Первые 12 строк уточняющей группировки по десятилетию и жанру:")
segment_preview = select_table("""
    SELECT
        release_decade,
        genre,
        films_count,
        complete_fin_count,
        recouped_count,
        round(median_return_ratio, 2) AS median_return_ratio,
        round(recoup_rate_pct, 2) AS recoup_rate_pct,
        round(median_imdb_score, 2) AS median_imdb_score
    FROM segment_metrics
    ORDER BY release_decade, films_count DESC, genre
    LIMIT 12
""")
print("Сводка по четырём полным десятилетиям выпуска:")
decade_totals = select_table("""
    SELECT
        release_decade,
        count(*) AS films_count,
        count(*) FILTER (WHERE NOT incomplete_financials_flag) AS complete_fin_count,
        count(*) FILTER (WHERE is_gross_ge_budget) AS recouped_count,
        round(median(gross_to_budget_ratio), 2) AS median_return_ratio,
        round(
            100.0 * count(*) FILTER (WHERE is_gross_ge_budget)
            / nullif(count(*) FILTER (WHERE NOT incomplete_financials_flag), 0),
            2
        ) AS recoup_rate_pct,
        round(median(imdb_score), 2) AS median_imdb_score
    FROM analysis_base
    GROUP BY release_decade
    ORDER BY release_decade
""")
segment_rows = connection.execute("""
    SELECT *
    FROM segment_metrics
    ORDER BY release_decade, films_count DESC, genre
""").fetchall()
print(f"Всего строк уточняющей группировки (десятилетие × жанр): {len(segment_rows)}")

connection.execute("""
    CREATE OR REPLACE TEMP TABLE distribution_values AS
    SELECT
        movie_title,
        release_year,
        release_decade,
        genre,
        imdb_score,
        budget_usd,
        gross_usd,
        gross_to_budget_ratio,
        incomplete_financials_flag
    FROM analysis_base
""")
distribution_summary = select_table("""
    SELECT
        count(*) AS total_rows,
        count(imdb_score) AS score_count,
        min(imdb_score) AS min_score,
        round(median(imdb_score), 2) AS median_score,
        max(imdb_score) AS max_score,
        count(gross_to_budget_ratio) AS ratio_count,
        round(min(gross_to_budget_ratio), 6) AS min_ratio,
        round(median(gross_to_budget_ratio), 2) AS median_ratio,
        round(max(gross_to_budget_ratio), 2) AS max_ratio
    FROM distribution_values
""")[0]
imdb_scores = [
    row[0]
    for row in connection.execute("""
        SELECT imdb_score
        FROM distribution_values
        ORDER BY imdb_score
    """).fetchall()
]
return_ratios = [
    row[0]
    for row in connection.execute("""
        SELECT gross_to_budget_ratio
        FROM distribution_values
        WHERE NOT incomplete_financials_flag
        ORDER BY gross_to_budget_ratio
    """).fetchall()
]
paired_fin_scores = connection.execute("""
    SELECT budget_usd, gross_usd, gross_to_budget_ratio, imdb_score
    FROM distribution_values
    WHERE NOT incomplete_financials_flag
    ORDER BY release_year, lower(movie_title)
""").fetchall()

Первые 12 строк уточняющей группировки по десятилетию и жанру:


release_decade,genre,films_count,complete_fin_count,recouped_count,median_return_ratio,recoup_rate_pct,median_imdb_score
1980-1989,Comedy,565,278,193,1.96,69.42,6.2
1980-1989,Action,320,222,140,1.56,63.06,6.0
1980-1989,Drama,312,125,61,0.93,48.8,6.55
1980-1989,Adventure,105,72,38,1.36,52.78,6.3
1980-1989,Crime,99,60,30,0.99,50.0,6.4
1980-1989,Horror,95,60,46,2.55,76.67,5.8
1980-1989,Biography,66,33,21,1.47,63.64,7.0
1980-1989,Animation,37,19,13,2.39,68.42,7.1
1980-1989,Fantasy,16,14,10,2.78,71.43,6.2
1980-1989,Thriller,10,2,1,1.05,50.0,6.15


Строк результата: 12
Сводка по четырём полным десятилетиям выпуска:


release_decade,films_count,complete_fin_count,recouped_count,median_return_ratio,recoup_rate_pct,median_imdb_score
1980-1989,1643,891,555,1.57,62.29,6.3
1990-1999,2000,1370,737,1.15,53.8,6.5
2000-2009,2000,1651,1182,1.88,71.59,6.5
2010-2019,1997,1515,1203,2.55,79.41,6.5


Строк результата: 4
Всего строк уточняющей группировки (десятилетие × жанр): 62


total_rows,score_count,min_score,median_score,max_score,ratio_count,min_ratio,median_ratio,max_ratio
7640,7640,1.9,6.5,9.3,5427,2.1e-05,1.83,12890.39


Строк результата: 1


<a id="validation"></a>
## 8. Проверка расчётов

Проверяем сходимость количества фильмов и размера финансовой подвыборки между таблицами, равенство суммы долей жанров `100 %`, совпадение общей доли окупаемости при расчёте по строкам и через отношение сумм числителей и знаменателей групп (`100.0 * sum(recouped_count) / nullif(sum(complete_fin_count), 0)`), а также полноту индивидуальных выборок в `distribution_values`.

In [7]:
# M07. Проверки
check_rows = select_table("""
    WITH checks AS (
        SELECT
            1 AS check_no,
            'Сумма количества фильмов по жанрам равна общему итогу' AS check_name,
            (SELECT sum(films_count) FROM genre_metrics)::DOUBLE AS actual,
            (SELECT films_count FROM overall_metrics)::DOUBLE AS expected
        UNION ALL
        SELECT
            2,
            'Сумма количества фильмов по десятилетиям и жанрам равна общему итогу',
            (SELECT sum(films_count) FROM segment_metrics)::DOUBLE,
            (SELECT films_count FROM overall_metrics)::DOUBLE
        UNION ALL
        SELECT
            3,
            'Сумма полных финансовых пар по сегментам совпадает с отбором по флагу',
            (SELECT sum(complete_fin_count) FROM segment_metrics)::DOUBLE,
            (
                SELECT count(*)
                FROM analysis_base
                WHERE NOT incomplete_financials_flag
            )::DOUBLE
        UNION ALL
        SELECT
            4,
            'Сумма долей жанров по оконной функции даёт 100 %',
            (SELECT sum(percent_of_films) FROM genre_metrics)::DOUBLE,
            100.0::DOUBLE
        UNION ALL
        SELECT
            5,
            'Общая доля окупаемости совпадает при расчёте по строкам и через суммы групп',
            (
                SELECT 100.0 * sum(recouped_count) / nullif(sum(complete_fin_count), 0)
                FROM genre_metrics
            )::DOUBLE,
            (SELECT recoup_rate_pct FROM overall_metrics)::DOUBLE
        UNION ALL
        SELECT
            6,
            'Число оценок IMDb в distribution_values равно общему числу фильмов',
            (SELECT count(imdb_score) FROM distribution_values)::DOUBLE,
            (SELECT films_count FROM overall_metrics)::DOUBLE
        UNION ALL
        SELECT
            7,
            'Число коэффициентов отдачи в distribution_values равно числу полных финансовых пар',
            (SELECT count(gross_to_budget_ratio) FROM distribution_values)::DOUBLE,
            (SELECT complete_fin_count FROM overall_metrics)::DOUBLE
    )
    SELECT
        check_name,
        round(actual, 2) AS actual,
        round(expected, 2) AS expected,
        abs(actual - expected) < 1e-4 AS passed
    FROM checks
    ORDER BY check_no
""")
assert all(row[3] for row in check_rows), "Проверки не сошлись: вернитесь к расчётам"
print(f"Проверок пройдено: {len(check_rows)} из {len(check_rows)}")

check_name,actual,expected,passed
Сумма количества фильмов по жанрам равна общему итогу,7640.0,7640.0,True
Сумма количества фильмов по десятилетиям и жанрам равна общему итогу,7640.0,7640.0,True
Сумма полных финансовых пар по сегментам совпадает с отбором по флагу,5427.0,5427.0,True
Сумма долей жанров по оконной функции даёт 100 %,100.0,100.0,True
Общая доля окупаемости совпадает при расчёте по строкам и через суммы групп,67.75,67.75,True
Число оценок IMDb в distribution_values равно общему числу фильмов,7640.0,7640.0,True
Число коэффициентов отдачи в distribution_values равно числу полных финансовых пар,5427.0,5427.0,True


Строк результата: 7
Проверок пройдено: 7 из 7


<a id="continuation"></a>
## Продолжение в ЛР № 7

Первая часть завершена. Определения метрик, описание подготовленных таблиц, результаты проверок и содержательный вывод оформлены в [docs/metrics.md](../docs/metrics.md).

В оперативной памяти DuckDB подготовлены временные таблицы:
- `overall_metrics` (`overall_raw`) — общие показатели по всем `7 640` фильмам;
- `genre_metrics` (`genre_raw`, `19` строк) — основная группировка по жанрам с оконной долей `percent_of_films`;
- `segment_metrics` (`segment_rows`, `62` строки) — уточняющая группировка по десятилетиям и жанрам (плюс сводка по десятилетиям `decade_totals`);
- `distribution_values` (`7 640` строк) и неокруглённые списки Python `imdb_scores` (`7 640` значений), `return_ratios` (`5 427` значений) и `paired_fin_scores` (`5 427` кортежей) — для построения распределений и диаграмм связи в ЛР № 7.

После перезапуска ядра выполните ячейки `M01–M07`: все таблицы и переменные полностью восстанавливаются из кода.